# cAPTure: Gate-0 data audit

Run this notebook in Google Colab with a **CPU runtime**. It audits development scenarios before selecting features, graph endpoints, or a window duration. It does not train models or access final-test scenarios.

The workflow uses the authors' public Drive for source CSVs and your mounted Drive for durable results, and local Colab disk for conversion and SQL aggregation. Process one scenario at a time. Start with `SMOKE`; `FULL_DEV` requires a reviewed smoke run.

The audit Parquet preserves every source row and raw column alongside diagnostic metadata. It is **not** a frozen, model-ready canonical dataset. No features are selected, labels guessed, or duplicate packets removed.


## 1. Mount Drive and load the project

Before running, push the implementation to the configured Git branch, or place an updated repository copy in Drive and set `PROJECT_ROOT` to that path. This notebook imports the repository module; it is not self-contained.

The manifest contains verified development CSV IDs, filenames, and byte sizes from the authors' [pre-merged training scenario folder](https://drive.google.com/drive/folders/1f8koJbuPs0_CWoItWdt3VF-lZ_R6JAms). No raw CSV copies are needed in your Drive. Colab downloads individual files by ID; it never recursively downloads the folder.

Folder metadata verification does not prove packet-level integrity. That is the purpose of Gate 0.


In [5]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
LOCAL_ROOT = Path("/content/capture_gate0_work")

if not PROJECT_ROOT.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)], check=True
    )

required_files = [
    PROJECT_ROOT / "code/python/utils/capture_data.py",
    PROJECT_ROOT / "code/python/requirements-capture.txt",
    PROJECT_ROOT / "configs/capture_experiment_v1.yaml",
]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Update the repository copy first: {missing_files}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture.txt")], check=True
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
print("CPU audit environment is ready.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CPU audit environment is ready.


## 2. Run the small synthetic checks

Run these checks before downloading or processing large files. They cover half-open windows, repeated packets, cross-chunk timestamp inversions, unknown labels, held-out exclusion, and smoke-review integrity. They use temporary synthetic data and do not inspect cAPTure contents.

These checks have not been executed in the development workspace; this Colab run is the first runtime validation.


In [6]:
test_environment = dict(os.environ)
test_environment["PYTHONPATH"] = str(PROJECT_ROOT / "code/python")
subprocess.run(
    [sys.executable, "-m", "unittest", "discover",
     "-s", str(PROJECT_ROOT / "code/python/tests"),
     "-p", "test_capture_data.py", "-v"],
    env=test_environment, cwd=PROJECT_ROOT, check=True,
)


CompletedProcess(args=['/usr/bin/python3', '-m', 'unittest', 'discover', '-s', '/content/temporalgnn-nids/code/python/tests', '-p', 'test_capture_data.py', '-v'], returncode=0)

## 3. Configure the audit and official downloads

Start with `SMOKE` (`train_empty_conn` and `train_dollar_char`). The source configuration is built from the manifest. Only development author-train scenarios are permitted. Your Drive stores reports and compressed audit Parquet; raw CSVs are staged on local Colab disk.

For `FULL_DEV`, supply a smoke-review file from section 8. A unique run directory prevents overwriting earlier results. Leave the chunk size at 25,000 initially; reduce it if wide CSV columns exhaust RAM. DuckDB is limited to two threads and 2 GB, but pandas and Arrow also need memory.

Google Drive may temporarily impose download quotas. If a download fails, the error is preserved; no alternative dataset is substituted.


In [7]:
from datetime import datetime, timezone
import json
import shutil
import pandas as pd
from IPython.display import display
from utils.capture_data import (
    AuditSchema, inspect_csv, load_manifest, run_gate0, selected_scenarios, stage_source,
    sha256_file, validate_smoke_review, write_json,
)

MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
MANIFEST = load_manifest(MANIFEST_PATH)
MODE = "SMOKE"
SCENARIOS = selected_scenarios(MANIFEST, MODE)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_" + MODE.lower()
DRIVE_RUN_DIR = DRIVE_ROOT / "runs" / RUN_ID

CHUNK_SIZE = 25_000
DUCKDB_MEMORY_LIMIT = "2GB"
DUCKDB_THREADS = 2
KEEP_AUDIT_PARQUET = True
SMOKE_REVIEW_PATH = None  # Set the reviewed smoke JSON path before FULL_DEV.

SOURCE_CACHE_ROOT = LOCAL_ROOT / RUN_ID / "sources"
SOURCES = {}
for scenario in SCENARIOS:
    entry = MANIFEST["scenarios"][scenario]
    required = ("source_file_id", "expected_filename", "expected_size_bytes")
    if any(entry.get(key) is None for key in required):
        raise ValueError(f"Source metadata is incomplete for {scenario}. Update the manifest.")
    SOURCES[scenario] = {
        **{key: entry[key] for key in required},
        "metadata_verified": True,
    }

if MODE == "FULL_DEV":
    if SMOKE_REVIEW_PATH is None:
        raise ValueError("Set SMOKE_REVIEW_PATH before inspecting FULL_DEV sources.")
    validate_smoke_review(Path(SMOKE_REVIEW_PATH), sha256_file(MANIFEST_PATH), MANIFEST)

print(f"Mode: {MODE}")
print(f"Scenarios: {SCENARIOS}")
print(f"Output directory: {DRIVE_RUN_DIR}")
print(f"Free local storage: {shutil.disk_usage(LOCAL_ROOT).free / 1024**3:.1f} GiB")


Mode: SMOKE
Scenarios: ['train_empty_conn', 'train_dollar_char']
Output directory: /content/drive/MyDrive/capture_gate0/runs/20260917T155516_689141Z_smoke
Free local storage: 86.4 GiB


## 4. Download and inspect the first scenario

Download the first selected CSV to local Colab disk, then read a prefix of 2,000 rows. The completed download is bound to its source configuration and a local SHA-256 receipt. The audit reuses it after verifying the receipt, so it is not downloaded twice.

Only one scenario is staged for preview. Remaining schemas are checked against their CSV headers when each scenario is processed. Missing attack labels in the preview do not mean the full scenario contains no attacks.

All raw columns are initially read as strings to preserve identifiers, leading zeros, empty fields, and mixed types. The preview is not used to fit preprocessing or select features.


In [8]:
CSV_SEPARATOR = ","
CSV_ENCODING = "utf-8-sig"
for source in SOURCES.values():
    source["separator"] = CSV_SEPARATOR
    source["encoding"] = CSV_ENCODING

PREVIEW_SCENARIO = SCENARIOS[0]
preview_path = stage_source(
    SOURCES[PREVIEW_SCENARIO], SOURCE_CACHE_ROOT / PREVIEW_SCENARIO,
)
inspection = inspect_csv(
    preview_path, separator=CSV_SEPARATOR,
    encoding=CSV_ENCODING, sample_rows=2_000,
)
INSPECTIONS = {PREVIEW_SCENARIO: inspection}
print(f"{PREVIEW_SCENARIO}: {inspection['source_size_bytes'] / 1024**3:.2f} GiB")
display(pd.DataFrame([
    {"column": column, "prefix_examples": values}
    for column, values in inspection["sample_values"].items()
]))


Downloading...
From (original): https://drive.google.com/uc?id=1nfR1RRZO3jMG_A9KDY8wyr0A0rVlHiBN
From (redirected): https://drive.google.com/uc?id=1nfR1RRZO3jMG_A9KDY8wyr0A0rVlHiBN&confirm=t&uuid=cfc31f11-cc39-4bc3-a5b0-5efdfeb4f8e9
To: /content/capture_gate0_work/20260917T155516_689141Z_smoke/sources/train_empty_conn/normal_empty_conn_train.csv.part
100%|██████████| 1.03G/1.03G [00:23<00:00, 43.7MB/s]


train_empty_conn: 0.96 GiB


,column,prefix_examples
0,layers_frame_frame.section_number,[1]
1,timestamp,"[1970-01-01 01:00:00.019792+01:00, 1970-01-01 ..."
2,layers_frame_frame.time_epoch,"[0.019792, 0.036245, 0.03809, 0.051729, 0.0682..."
3,layers_frame_frame.number,"[1, 2, 3, 4, 5, 6, 7, 8]"
4,layers_frame_frame.len,"[114, 90, 111, 64, 78, 70, 100, 74]"
...,...,...
103,phase_idx,[]
104,phase_name,[]
105,phase_number,[]
106,step_number,[]


## 5. Confirm the declared schema

The mapping below follows the authors' scenario-construction notebooks:

- `timestamp` is the post-merge timestamp. The relative frame epoch is not used because the merge adjusts `timestamp`.
- `label` contains `normal` or the attack command. It is also the most specific published attack-step name.
- `phase_name` and `sequence_id` retain the authors' evaluation annotations.
- Ethernet source and destination MAC fields are provisional Gate-0 endpoints because they cover IP, IPv6, ARP, and other Ethernet traffic. Gate 0 will determine whether this remains the graph node key.

The raw-to-binary mappings enumerate the values reported by the official merge notebooks for all five development author-train scenarios. Any additional value remains unmapped and blocks progression. Endpoint identifiers are topology metadata and are not approved as model features.

The diagnostic iteration key is `(scenario, attack_step, phase, sequence_id)`. Its scientific meaning must still be reviewed after the audit.


In [14]:
COMMON_SCHEMA = {
    "timestamp": "timestamp",
    "timestamp_unit": "datetime",
    "label": "label",
    "source_endpoint": "layers_eth_eth.src",
    "destination_endpoint": "layers_eth_eth.dst",
    "attack_step": "label",
    "phase": "phase_name",
    "sequence_id": "sequence_id",
    "separator": CSV_SEPARATOR,
    "encoding": CSV_ENCODING,
}
SCHEMA_OVERRIDES = {
    "train_empty_conn": {
        "label_mapping": {
            "normal": 0,
            "nmap_10_T4": 1,
            "brute_force_timing": 1,
            "empty_conn_ddos": 1,
            "nmap_mqtt": 1,
            "nmap_banner": 1,
            "empty_conn": 1,
            "nmap_sub": 1,
            "mqtt_cat": 1,
            "sftp_inst": 1,
        },
    },
    "train_dollar_char": {
        "label_mapping": {
            "normal": 0,
            "dollar_char": 1,
            "nmap_10_T5": 1,
            "nmap_mqtt": 1,
            "brute_force_malformed": 1,
            "nmap_banner": 1,
            "nmap_sub": 1,
            "mqtt_cat": 1,
            "scp_inst": 1,
        },
    },
    "train_qos_mid": {
        "label_mapping": {
            "normal": 0,
            "nmap_10_T5": 1,
            "qos_mid_ddos": 1,
            "brute_force_malformed": 1,
            "qos_mid": 1,
            "nmap_sub": 1,
            "nmap_mqtt": 1,
            "nmap_banner": 1,
            "mqtt_cat": 1,
            "scp_inst": 1,
        },
    },
    "train_slash_char": {
        "label_mapping": {
            "normal": 0,
            "nmap_10_T4": 1,
            "slash_char": 1,
            "nmap_10_T5": 1,
            "brute_force_timing": 1,
            "nmap_mqtt": 1,
            "nmap_banner": 1,
            "nmap_sub": 1,
            "mqtt_cat": 1,
            "sftp_inst": 1,
        },
    },
    "train_sub_exf": {
        "label_mapping": {
            "normal": 0,
            "nmap_10_T5": 1,
            "brute_force_timing": 1,
            "nmap_banner": 1,
            "nmap_sub": 1,
            "scp_exf": 1,
            "nmap_mqtt": 1,
            "mqtt_cat": 1,
        },
    },
}
SCHEMAS = {}
for scenario in SCENARIOS:
    settings = {**COMMON_SCHEMA, **SCHEMA_OVERRIDES.get(scenario, {})}
    unresolved = [key for key, value in settings.items() if value is None]
    if unresolved or not settings.get("label_mapping"):
        raise ValueError(f"Resolve schema fields for {scenario}: {unresolved}")
    schema = AuditSchema(**settings)
    if scenario in INSPECTIONS:
        schema.validate(INSPECTIONS[scenario]["columns"])
    SCHEMAS[scenario] = schema
print("All selected scenario mappings are explicit.")


All selected scenario mappings are explicit.


## 6. Run the complete scenario audit

This cell scans each selected CSV completely. It downloads or reuses one raw file locally, writes compressed audit Parquet, and computes disk-backed statistics for 1, 5, 10, and 30 seconds, including half-window origin shifts. Epoch-aligned boundaries are diagnostic candidates, not a frozen window choice.

Repeated packets remain distinct rows. Duplicate-row and duplicate-column detection uses SHA-256 fingerprints. Numeric parse failures may indicate legitimate categorical fields; inspect them before declaring malformed data. Window distributions describe occupied windows; empty windows between the first and last packet are counted separately.

Report files are copied to Drive and checksum-verified. Only then is that scenario's isolated local workspace deleted. The authors' originals are never modified. The preview download is reused and removed only after its results are safely persisted. A failure retains local work and records its location on Drive; a data-integrity blocker stops before the next scenario. SQL spill space can exceed raw size, so monitor local storage.

An audit with no automatic blockers still requires review of endpoints, sequence semantics, timestamps, duplicates, feature leakage, and window feasibility.


In [10]:
RESULTS = run_gate0(
    manifest_path=MANIFEST_PATH,
    mode=MODE,
    sources=SOURCES,
    schemas=SCHEMAS,
    local_root=LOCAL_ROOT,
    drive_run_dir=DRIVE_RUN_DIR,
    smoke_review_path=Path(SMOKE_REVIEW_PATH) if SMOKE_REVIEW_PATH else None,
    chunksize=CHUNK_SIZE,
    memory_limit=DUCKDB_MEMORY_LIMIT,
    threads=DUCKDB_THREADS,
    keep_audit_parquet=KEEP_AUDIT_PARQUET,
    source_cache_root=SOURCE_CACHE_ROOT,
)
print(json.dumps(RESULTS, indent=2))


Starting train_empty_conn. Local workspace: /content/capture_gate0_work/train_empty_conn_k9stw_vv
Reusing verified local source: normal_empty_conn_train.csv
train_empty_conn: converted 25,000 packets
train_empty_conn: converted 50,000 packets
train_empty_conn: converted 75,000 packets
train_empty_conn: converted 100,000 packets
train_empty_conn: converted 125,000 packets
train_empty_conn: converted 150,000 packets
train_empty_conn: converted 175,000 packets
train_empty_conn: converted 200,000 packets
train_empty_conn: converted 225,000 packets
train_empty_conn: converted 250,000 packets
train_empty_conn: converted 275,000 packets
train_empty_conn: converted 300,000 packets
train_empty_conn: converted 325,000 packets
train_empty_conn: converted 350,000 packets
train_empty_conn: converted 375,000 packets
train_empty_conn: converted 400,000 packets
train_empty_conn: converted 425,000 packets
train_empty_conn: converted 450,000 packets
train_empty_conn: converted 475,000 packets
train_empt

Downloading...
From (original): https://drive.google.com/uc?id=1N3IcqQz6tQe-iuvoqWWwVarPCJDK_cpe
From (redirected): https://drive.google.com/uc?id=1N3IcqQz6tQe-iuvoqWWwVarPCJDK_cpe&confirm=t&uuid=6bd60716-00ce-49e5-8619-e01cef0d4c73
To: /content/capture_gate0_work/20260917T155516_689141Z_smoke/sources/train_dollar_char/normal_dollar_char_train.csv.part
100%|██████████| 1.97G/1.97G [01:23<00:00, 23.5MB/s]


train_dollar_char: converted 25,000 packets
train_dollar_char: converted 50,000 packets
train_dollar_char: converted 75,000 packets
train_dollar_char: converted 100,000 packets
train_dollar_char: converted 125,000 packets
train_dollar_char: converted 150,000 packets
train_dollar_char: converted 175,000 packets
train_dollar_char: converted 200,000 packets
train_dollar_char: converted 225,000 packets
train_dollar_char: converted 250,000 packets
train_dollar_char: converted 275,000 packets
train_dollar_char: converted 300,000 packets
train_dollar_char: converted 325,000 packets
train_dollar_char: converted 350,000 packets
train_dollar_char: converted 375,000 packets
train_dollar_char: converted 400,000 packets
train_dollar_char: converted 425,000 packets
train_dollar_char: converted 450,000 packets
train_dollar_char: converted 475,000 packets
train_dollar_char: converted 500,000 packets
train_dollar_char: converted 525,000 packets
train_dollar_char: converted 550,000 packets
train_dollar_

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved and verified train_dollar_char on Drive. Removed its temporary local copy.
{
  "train_empty_conn": {
    "status": "review_required",
    "blockers": [],
    "report": "/content/drive/MyDrive/capture_gate0/runs/20260917T155516_689141Z_smoke/train_empty_conn/audit_report.json"
  },
  "train_dollar_char": {
    "status": "review_required",
    "blockers": [],
    "report": "/content/drive/MyDrive/capture_gate0/runs/20260917T155516_689141Z_smoke/train_dollar_char/audit_report.json"
  }
}


## 7. Review the saved reports

The run directory contains the manifest snapshot, resolved runtime configuration, Git provenance, run status, and per-scenario artifacts:

- `audit_report.json`: schema, packet counts, raw labels, column diagnostics, blockers, and window summaries.
- `packets.audit.parquet`: all rows and raw fields, plus canonical diagnostic metadata (optional durable retention).
- `attack_iterations.parquet`: counts and durations grouped by attack step, phase, and sequence.
- `windows_*s_offset_*s.parquet`: occupied-window packet, node, and directed-pair counts.
- `artifact_checksums.json`: hashes of the persisted files.

A zero-duration attack iteration is possible when all its packets have the same timestamp. Duplicate benign data across scenarios sharing the same benign source is not removed; the manifest's background-separated folds address that evaluation risk.


In [11]:
REPORTS = {}
for scenario, result in RESULTS.items():
    report = json.loads(Path(result["report"]).read_text())
    REPORTS[scenario] = report
    print(f"\n{scenario}: {report['status']}")
    print(f"Blockers: {report['blockers']}")
    display(pd.DataFrame([report["counts"]]))
    display(pd.DataFrame(report["raw_labels"]))
    window_summary = pd.DataFrame(report["windows"])
    window_summary["mean_packets_per_second_occupied"] = (
        window_summary["mean_packets_occupied"] / window_summary["width_seconds"]
    )
    display(window_summary)
    display(pd.DataFrame(report["column_profiles"]).T)
    print("Duplicate column groups:", report["duplicate_column_groups_sha256"])
    iterations = pd.read_parquet(Path(result["report"]).parent / "attack_iterations.parquet")
    display(iterations.head(50))



train_empty_conn: review_required
Blockers: []


,packets,normal_packets,attack_packets,unmapped_labels,invalid_timestamps,missing_endpoints,incomplete_attack_annotations,first_timestamp_seconds,last_timestamp_seconds,duplicate_raw_rows_sha256,unique_endpoints,unique_directed_pairs
0,1175779,746806,428973,0,0,0,0,0.019792,46799.813163,0,160,623


,raw_label,packets
0,normal,746806
1,nmap_10_T4,255667
2,brute_force_timing,107515
3,empty_conn_ddos,28269
4,nmap_mqtt,14168
5,nmap_banner,12649
6,empty_conn,6650
7,nmap_sub,2277
8,mqtt_cat,1171
9,sftp_inst,607


,width_seconds,origin_offset_seconds,artifact,occupied_windows,empty_windows_between_first_and_last,mean_packets_occupied,max_packets,packet_quantiles_occupied,max_nodes,node_quantiles_occupied,max_directed_pairs,pair_quantiles_occupied,mean_packets_per_second_occupied
0,1,0.0,windows_1s_offset_0s.parquet,44480,2320,26.433880,15524,"[16.0, 37.0, 145.41999999999825]",56,"[4.0, 8.0, 10.0]",96,"[5.0, 11.0, 14.0]",26.433880
1,1,0.5,windows_1s_offset_0.5s.parquet,44435,2366,26.460650,15616,"[16.0, 37.0, 146.0]",67,"[4.0, 8.0, 11.0]",92,"[5.0, 11.0, 14.659999999996217]",26.460650
2,5,0.0,windows_5s_offset_0s.parquet,9360,0,125.617415,42145,"[79.0, 134.04999999999927, 616.4099999999999]",86,"[13.0, 20.0, 33.0]",108,"[20.0, 31.0, 40.409999999999854]",25.123483
3,5,2.5,windows_5s_offset_2.5s.parquet,9361,0,125.603995,42148,"[79.0, 132.0, 616.1999999999989]",80,"[13.0, 20.0, 33.0]",106,"[20.0, 31.0, 40.0]",25.120799
4,10,0.0,windows_10s_offset_0s.parquet,4680,0,251.234829,42245,"[159.0, 261.10000000000036, 920.0]",88,"[15.0, 22.0, 37.0]",111,"[24.0, 37.0, 62.0]",25.123483
5,10,5.0,windows_10s_offset_5s.parquet,4681,0,251.181158,42231,"[161.0, 267.0, 874.1999999999998]",86,"[15.0, 22.0, 37.0]",115,"[24.0, 36.0, 60.19999999999982]",25.118116
6,30,0.0,windows_30s_offset_0s.parquet,1560,0,753.704487,42592,"[490.5, 1032.1, 1678.4400000000069]",95,"[19.0, 37.0, 45.0]",162,"[30.0, 52.0, 74.23000000000025]",25.123483
7,30,15.0,windows_30s_offset_15s.parquet,1561,0,753.221653,43255,"[492.0, 1027.0, 1633.0000000000018]",93,"[19.0, 38.0, 45.40000000000009]",132,"[31.0, 53.0, 77.0]",25.107388


,missing,numeric_values,constant_nonmissing,numeric_parse_failures_nonmissing,interpretation
layers_frame_frame.section_number,0,1175779,True,0,Parse failures may be valid categorical values...
timestamp,0,0,False,1175779,Parse failures may be valid categorical values...
layers_frame_frame.time_epoch,0,1175779,False,0,Parse failures may be valid categorical values...
layers_frame_frame.number,0,1175779,False,0,Parse failures may be valid categorical values...
layers_frame_frame.len,0,1175779,False,0,Parse failures may be valid categorical values...
...,...,...,...,...,...
phase_idx,746806,428973,False,0,Parse failures may be valid categorical values...
phase_name,746806,0,False,428973,Parse failures may be valid categorical values...
phase_number,746806,428973,False,0,Parse failures may be valid categorical values...
step_number,746806,428973,False,0,Parse failures may be valid categorical values...


...

,raw_label,packets
0,normal,1705003
1,dollar_char,739943
2,nmap_10_T5,383310
3,nmap_mqtt,19736
4,brute_force_malformed,15561
5,nmap_banner,12635
6,nmap_sub,5333
7,mqtt_cat,655
8,scp_inst,379


,width_seconds,origin_offset_seconds,artifact,occupied_windows,empty_windows_between_first_and_last,mean_packets_occupied,max_packets,packet_quantiles_occupied,max_nodes,node_quantiles_occupied,max_directed_pairs,pair_quantiles_occupied,mean_packets_per_second_occupied
0,1,0.0,windows_1s_offset_0s.parquet,40000,6797,72.063875,15360,"[38.0, 168.0, 384.0]",57,"[6.0, 11.0, 13.0]",94,"[9.0, 16.0, 20.0]",72.063875
1,1,0.5,windows_1s_offset_0.5s.parquet,40000,6798,72.063875,15850,"[38.0, 168.0, 388.0200000000041]",68,"[6.0, 11.0, 13.0]",94,"[9.0, 16.0, 19.0]",72.063875
2,5,0.0,windows_5s_offset_0s.parquet,9117,243,316.173632,43211,"[189.0, 716.0, 1094.8400000000001]",85,"[13.0, 20.0, 35.0]",112,"[21.0, 32.0, 48.840000000000146]",63.234726
3,5,2.5,windows_5s_offset_2.5s.parquet,9119,241,316.104288,42436,"[189.0, 713.0, 1067.9199999999983]",79,"[13.0, 20.0, 34.0]",105,"[21.0, 32.0, 46.0]",63.220858
4,10,0.0,windows_10s_offset_0s.parquet,4680,0,615.930556,43458,"[415.0, 1354.0500000000002, 1914.5200000000004]",96,"[15.0, 24.0, 39.0]",123,"[25.0, 40.0, 69.21000000000004]",61.593056
5,10,5.0,windows_10s_offset_5s.parquet,4681,0,615.798975,43477,"[410.0, 1351.0, 1850.199999999998]",85,"[16.0, 24.0, 39.0]",112,"[25.0, 40.0, 68.0]",61.579897
6,30,0.0,windows_30s_offset_0s.parquet,1560,0,1847.791667,44273,"[1426.0, 3814.05, 5273.710000000003]",97,"[20.0, 41.0, 46.0]",170,"[34.0, 62.0, 84.41000000000008]",61.593056
7,30,15.0,windows_30s_offset_15s.parquet,1561,0,1846.607944,44876,"[1401.0, 3813.0, 4887.400000000016]",97,"[20.0, 40.0, 46.0]",131,"[33.0, 61.0, 83.0]",61.553598


,missing,numeric_values,constant_nonmissing,numeric_parse_failures_nonmissing,interpretation
layers_frame_frame.section_number,0,2882555,True,0,Parse failures may be valid categorical values...
timestamp,0,0,False,2882555,Parse failures may be valid categorical values...
layers_frame_frame.time_epoch,0,2882555,False,0,Parse failures may be valid categorical values...
layers_frame_frame.number,0,2882555,False,0,Parse failures may be valid categorical values...
layers_frame_frame.len,0,2882555,False,0,Parse failures may be valid categorical values...
...,...,...,...,...,...
phase_idx,1705003,1177552,False,0,Parse failures may be valid categorical values...
phase_name,1705003,0,False,1177552,Parse failures may be valid categorical values...
phase_number,1705003,1177552,False,0,Parse failures may be valid categorical values...
step_number,1705003,1177552,False,0,Parse failures may be valid categorical values...


Duplicate column groups: [['layers_frame_frame.len', 'layers_frame_frame.cap_len'], ['layers_eth_eth.dst', 'layers_eth_eth.dst_tree_eth.addr'], ['layers_eth_eth.dst_tree_eth.dst_resolved', 'layers_eth_eth.dst_tree_eth.addr_resolved'], ['layers_eth_eth.dst_tree_eth.dst.oui', 'layers_eth_eth.dst_tree_eth.addr.oui'], ['layers_eth_eth.src', 'layers_eth_eth.src_tree_eth.src_resolved'], ['layers_eth_eth.src_tree_eth.src.oui', 'layers_eth_eth.src_tree_eth.addr.oui'], ['layers_ipv6_ipv6.addr', 'layers_ipv6_ipv6.host'], ['layers_ip_ip.addr', 'layers_ip_ip.host'], ['layers_tcp_tcp.dstport', 'layers_tcp_tcp.port'], ['layers_ssh_SSH Version 2_Key Exchange_Algorithms_ssh.encryption_algorithms_client_to_server', 'layers_ssh_SSH Version 2_Key Exchange_Algorithms_ssh.encryption_algorithms_server_to_client'], ['layers_ssh_SSH Version 2_Key Exchange_Algorithms_ssh.mac_algorithms_client_to_server', 'layers_ssh_SSH Version 2_Key Exchange_Algorithms_ssh.mac_algorithms_server_to_client'], ['layers_ssh_SSH V

,attack_step,phase,sequence_id,packets,start_seconds,end_seconds,duration_seconds
0,nmap_10_T5,RECONNAISSANCE,3.1,42629,3361.941748,3417.969939,56.028191
1,nmap_10_T5,RECONNAISSANCE,5.1,42581,3596.558603,3653.160577,56.601974
2,nmap_10_T5,RECONNAISSANCE,9.1,42577,4156.802665,4212.535711,55.733046
3,nmap_10_T5,RECONNAISSANCE,13.1,42580,4728.300644,4784.362588,56.061944
4,mqtt_cat,DISCOVERY,27.1,95,6335.281688,6340.503437,5.221749
5,mqtt_cat,DISCOVERY,36.1,91,6761.096528,6766.424700,5.328172
6,nmap_sub,DISCOVERY,38.1,760,6809.849659,6864.201538,54.351879
7,nmap_mqtt,DISCOVERY,56.1,964,8727.485915,8969.898472,242.412557
8,nmap_mqtt,DISCOVERY,58.1,965,9013.658513,9256.053345,242.394832
9,dollar_char,EXPLOIT,82.1,5376,10427.774340,10495.094551,67.320211


In [12]:
empty_columns = set(REPORTS["train_empty_conn"]["column_profiles"])
dollar_columns = set(REPORTS["train_dollar_char"]["column_profiles"])

print("Only in train_empty_conn:")
print(sorted(empty_columns - dollar_columns))

print("\nOnly in train_dollar_char:")
print(sorted(dollar_columns - empty_columns))

for scenario, report in REPORTS.items():
    print(f"\n{scenario}")
    print("Timestamp inversions:", report["timestamp_order_inversions"])

    artifact_directory = Path(RESULTS[scenario]["report"]).parent
    iterations = pd.read_parquet(
        artifact_directory / "attack_iterations.parquet"
    )

    sequence_cardinality = iterations.groupby("sequence_id").agg(
        attack_steps=("attack_step", "nunique"),
        phases=("phase", "nunique"),
    )
    conflicts = sequence_cardinality[
        (sequence_cardinality["attack_steps"] > 1)
        | (sequence_cardinality["phases"] > 1)
    ]

    print("Attack iterations:", len(iterations))
    print("Conflicting sequence identifiers:", len(conflicts))
    display(conflicts.head())

Only in train_empty_conn:
['layers_tcp_tcp.options_tree_tcp.options.sack']

Only in train_dollar_char:
[]

train_empty_conn
Timestamp inversions: 0
Attack iterations: 107
Conflicting sequence identifiers: 0


,attack_steps,phases
sequence_id,,



train_dollar_char
Timestamp inversions: 0
Attack iterations: 232
Conflicting sequence identifiers: 0


,attack_steps,phases
sequence_id,,


## 8. Record the smoke review before FULL_DEV

Do this only after both smoke scenarios finish and their scientific checks have been reviewed. Record the rationale and any unresolved choices. This review permits the full development audit; it does not freeze features, pass a modeling gate, or authorize final-test access.

The review binds the exact manifest and report hashes. Changing the manifest invalidates it. Keep `APPROVE_SMOKE=False` until review is complete. To run `FULL_DEV`, return to section 3, change the mode, use the five sources already configured in the manifest, and set `SMOKE_REVIEW_PATH` to the printed path.


In [16]:
APPROVE_SMOKE = True
REVIEW_NOTES = """
  SMOKE review passed for train_empty_conn and train_dollar_char.
  Both reports have no automatic blockers. Packet and label counts match
  the official scenario-construction notebooks. All labels are mapped,
  timestamps are valid and monotonic, Ethernet endpoints and attack
  annotations are complete, and no complete raw-row duplicates were found.
  Each sequence_id maps to exactly one attack step and phase.

  The only schema difference is
  layers_tcp_tcp.options_tree_tcp.options.sack, which is present only in
  train_empty_conn. It is treated as an optional protocol-derived column
  pending the full-development feature review. Window duration, endpoint
  policy, and model feature schema remain unfrozen.
  """


if APPROVE_SMOKE:
    if MODE != "SMOKE":
        raise ValueError("Create smoke reviews only from SMOKE runs.")
    expected = selected_scenarios(MANIFEST, "SMOKE")
    if set(REPORTS) != set(expected) or any(report["blockers"] for report in REPORTS.values()):
        raise ValueError("Both smoke reports must exist and have no automatic blockers.")
    if not REVIEW_NOTES.strip():
        raise ValueError("Record the review rationale before approval.")
    review = {
        "approved": True,
        "manifest_sha256": sha256_file(MANIFEST_PATH),
        "review_notes": REVIEW_NOTES,
        "reports": {
            scenario: {
                "path": RESULTS[scenario]["report"],
                "sha256": sha256_file(Path(RESULTS[scenario]["report"])),
            }
            for scenario in expected
        },
    }
    review_path = DRIVE_RUN_DIR / "smoke_review.json"
    if review_path.exists():
        raise FileExistsError("A smoke review already exists; preserve the original decision.")
    write_json(review_path, review)
    validate_smoke_review(review_path, sha256_file(MANIFEST_PATH), MANIFEST)
    print(f"Smoke review saved: {review_path}")
else:
    print("Smoke review remains pending. No approval was recorded.")


Smoke review saved: /content/drive/MyDrive/capture_gate0/runs/20260917T155516_689141Z_smoke/smoke_review.json
